# Laboratorio 3 - Indexacion y retrieval

**Duracion estimada:** 45 minutos

**Objetivo**

- Construir el pipeline completo de indexacion.
- Guardar chunks y metadatos en Qdrant.
- Validar que el retrieval devuelve contexto util.

**Prerequisitos**

- Laboratorios 1 y 2 revisados
- Stack Docker operativo
- Modelos de Ollama descargados

**Criterios de exito**

- Creas la coleccion final con el tamano correcto.
- Indexas chunks reales con metadatos utiles.
- Recuperas resultados razonables para una pregunta del dominio.


## Antes de empezar

Aqui unimos tres piezas:

1. documentos y chunking,
2. embeddings,
3. almacenamiento vectorial en Qdrant.


In [ ]:
import os
import uuid
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

load_dotenv(Path("..").resolve() / ".env")

OLLAMA_PORT = os.getenv("OLLAMA_PORT", "11434")
QDRANT_PORT = os.getenv("QDRANT_PORT", "6333")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "embeddinggemma")
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "frasohome_docs")
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "650"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "100"))

OLLAMA_BASE_URL = f"http://127.0.0.1:{OLLAMA_PORT}"
QDRANT_URL = f"http://127.0.0.1:{QDRANT_PORT}"
DOCS_DIR = Path("..").resolve() / "docs"

print(
    {
        "OLLAMA_BASE_URL": OLLAMA_BASE_URL,
        "QDRANT_URL": QDRANT_URL,
        "COLLECTION_NAME": COLLECTION_NAME,
        "CHUNK_SIZE": CHUNK_SIZE,
        "CHUNK_OVERLAP": CHUNK_OVERLAP,
    }
)


## Paso 1 - Cargar y dividir documentos

Reutiliza el patron del laboratorio anterior para cargar los `.md` y partirlos en chunks con `RecursiveCharacterTextSplitter`.

Asigna ademas un `chunk_id` consecutivo a cada trozo.


In [ ]:
loader = DirectoryLoader(
    str(DOCS_DIR),
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(documents)

for idx, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = idx

print({"documents": len(documents), "chunks": len(chunks)})


## Paso 2 - Generar embeddings e inspeccionar la dimension

Crea una instancia de `OllamaEmbeddings` y genera un vector por chunk.


In [ ]:
embedding_model = OllamaEmbeddings(
    model=EMBEDDING_MODEL,
    base_url=OLLAMA_BASE_URL,
)
texts = [chunk.page_content for chunk in chunks]
vectors = embedding_model.embed_documents(texts)
vector_size = len(vectors[0])

print({"chunks": len(chunks), "vectors": len(vectors), "vector_size": vector_size})


### Checkpoint 1

Verifica:

- el numero de embeddings coincide con el numero de chunks;
- el tamano del vector sera el que defina la coleccion;
- ya puedes explicar por que no se puede crear la coleccion final antes de conocer `vector_size`.


## Paso 3 - Crear la coleccion final e indexar puntos

Guarda en Qdrant cada chunk como un `PointStruct` con:

- `id` unico,
- `vector`,
- `source`,
- `chunk_id`,
- `text`.


In [ ]:
client = QdrantClient(url=QDRANT_URL)
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
)

points = []
for chunk, vector in zip(chunks, vectors):
    points.append(
        PointStruct(
            id=str(uuid.uuid4()),
            vector=vector,
            payload={
                "source": chunk.metadata.get("source", "desconocido"),
                "chunk_id": chunk.metadata["chunk_id"],
                "text": chunk.page_content,
            },
        )
    )

client.upsert(collection_name=COLLECTION_NAME, points=points)
client.get_collection(COLLECTION_NAME)


## Paso 4 - Probar retrieval sobre la coleccion indexada

Usa la pregunta:

`Que debo hacer si faltan tornillos en la caja?`

Genera el embedding de consulta y recupera los 3 mejores resultados.


In [ ]:
question = "Que debo hacer si faltan tornillos en la caja?"
query_vector = embedding_model.embed_query(question)
hits = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=3,
).points

for hit in hits:
    print("=" * 80)
    print(
        {
            "score": round(hit.score, 4),
            "source": hit.payload.get("source"),
            "chunk_id": hit.payload.get("chunk_id"),
        }
    )
    print(hit.payload.get("text", "")[:500])


### Checkpoint 2

Comprueba estos puntos:

- al menos un resultado relevante procede del documento correcto;
- el `payload` te deja auditar rapidamente la recuperacion;
- el texto del chunk recuperado contiene evidencia suficiente para responder.


## Reflexion 1

**Respuesta orientativa**

- Son imprescindibles metadatos como `source`, `chunk_id` y el propio `text`.
- Si solo guardas el vector, no puedes inspeccionar ni justificar por que el retrieval devolvio ese resultado.


## Reflexion 2

**Respuesta orientativa**

- Un resultado puede tener buen score y aun asi quedarse corto si el chunk no contiene toda la evidencia necesaria.
- Antes de generar conviene revisar manualmente `source`, `score` y contenido recuperado para ver si realmente sostienen la respuesta.

**Mini extension opcional**

Cambia la pregunta por una relacionada con devoluciones y compara las fuentes recuperadas.
